# Aggregating Attribute Data


In the previous section, we aggregated point data by polygon — counting how many features fall within each spatial unit. But a count is often not enough: the **attributes** of those features matter too, and they can be summarised.

In this section, we will aggregate attribute data from a layer of Vienna buildings by district.

- We will perform a spatial join to assign each building to its district.
- We will group the data by district and calculate statistics on the year of construction (mean, median, minimum, and maximum).
- We will join the results back to the polygon layer and visualise the output.


## 0. Importing Libraries and Preparing the Data


### 0.1. Importing Libraries


In [ ]:
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt  # plotting
import matplotlib as mpl         # core matplotlib module

### 0.2. Preparing the Data


This section uses two files from `data/vienna/`:

- **vienna_admin.gpkg** — boundaries of the city's districts and census districts;
- **vienna_buildings.csv** — the city's building register, with year of construction.

_Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._


We read the district boundaries of Vienna from the GeoPackage file.


In [ ]:
districts = gpd.read_file("../../data/vienna/vienna_admin.gpkg", layer="district")

districts.explore(tiles="cartodbpositron")

Then the building register.


In [ ]:
buildings_csv = pd.read_csv("../../data/vienna/vienna_buildings.csv")

# Preview the data to understand which fields contain the coordinates
buildings_csv.head()

Coordinates are stored in a single `SHAPE` field, written as text in **WKT** (Well-Known Text) — the standard textual notation for geometry, in which a point reads `POINT (longitude latitude)`. GeoPandas parses that notation directly, so there is no need to pull the numbers apart by hand.


In [ ]:
# GeoSeries.from_wkt turns the WKT strings into geometry.
# WKT always writes x (longitude) before y (latitude), so the order is unambiguous —
# unlike a plain pair of coordinate columns, where you have to check which is which.
buildings_gdf = gpd.GeoDataFrame(
    buildings_csv,
    geometry=gpd.GeoSeries.from_wkt(buildings_csv["SHAPE"]),
    crs="EPSG:4326"
)

The first 100 features on a map. Rendering the full dataset is not recommended — with nearly sixty thousand points the map may fail to load.


In [ ]:
buildings_gdf.head(100).explore(tiles="cartodbpositron")

And the summary information for the GeoDataFrame:


In [ ]:
buildings_gdf.info()

The column names are abbreviated German, but the ones we need are readable enough: `BAUJAHR` is the year of construction, `GESCH_ANZ` the number of storeys, `L_NUTZUNG` the type of use.

For this exercise, we will focus on `BAUJAHR`. For each district, we will calculate the mean, median, earliest, and latest year of construction across all buildings within it.


## 1. Spatial Join

First, we need to join the buildings layer with the district boundaries. We perform a spatial join to determine which district each building belongs to.


### 1.1. Checking the CRS


Do the two layers share a CRS?


In [ ]:
districts.crs == buildings_gdf.crs

They match, so we can proceed without reprojecting.


### 1.2. Performing the Spatial Join


For each building, we identify the district it falls within using `sjoin` with the `within` predicate. We use a left join (`how="left"`) to retain all buildings, including any that fall outside the district boundaries.


In [ ]:
buildings_in_district = gpd.sjoin(
    buildings_gdf,
    districts,
    how="left",
    predicate="within"
)

The spatial join appends the attributes of the district to each building.

The first five rows of the result:


In [ ]:
buildings_in_district.head()

## 2. Aggregating Attribute Data

Now that each building has been assigned to a district, we can analyse the attribute data. We will group buildings by district and compute summary statistics for the year of construction field.


### 2.1. Checking the Data

Before summarising a field, look at it. Check the type, how much of it is actually filled in, and whether the values are plausible.


In [ ]:
print(f"Data type: {buildings_in_district['BAUJAHR'].dtype}")
print(f"Filled in: {buildings_in_district['BAUJAHR'].notna().sum()} of {len(buildings_in_district)}")
print(f"Range: {buildings_in_district['BAUJAHR'].min():.0f}–{buildings_in_district['BAUJAHR'].max():.0f}")

Three things stand out.

The field is already numeric — pandas reads a column of numbers as `float64`, and it is `float` rather than `int` precisely because some values are missing. Missing values need no special handling below: the pandas aggregation methods skip them.

Only about a fifth of the buildings carry a year at all. That is enough for a district-level average, but it is worth knowing before you present the number.

And the earliest value is impossible — no building in Vienna dates from the year 190. Values like that are typos in the register, and left in place they drag both the minimum and the mean off. Let's discard the years that cannot be right.

In [ ]:
# Keep rows with a plausible year, and rows with no year at all —
# the latter are simply skipped by the statistics below.
plausible = buildings_in_district["BAUJAHR"].isna() | (buildings_in_district["BAUJAHR"] >= 1100)
buildings_in_district = buildings_in_district[plausible]

print(f"Range: {buildings_in_district['BAUJAHR'].min():.0f}–{buildings_in_district['BAUJAHR'].max():.0f}")

The range now runs from the medieval core to the late twentieth century, which is what we would expect. We can compute the statistics.


### 2.2. Computing Summary Statistics


Let's calculate the mean year of construction per district and join the result back to the polygon layer.

We write the result into a new layer, `district_stats`, and leave `districts` untouched — that way the cells below can be re-run in any order without joining the same columns twice.

In [ ]:
# Group by district and calculate mean year of construction
build_year_mean = buildings_in_district.groupby("NAME")["BAUJAHR"].mean()

# Join the result to the districts layer
district_stats = districts.merge(
    build_year_mean.rename("build_year_mean"),
    on="NAME",
    how="left"
)

The result on the map:

In [ ]:
district_stats.explore(column="build_year_mean", cmap="YlGnBu", tiles="cartodbpositron")

Now let's compute the median, earliest, and latest year of construction in the same way.


In [ ]:
# Calculate median, minimum, and maximum year of construction per district
build_year_median = buildings_in_district.groupby("NAME")["BAUJAHR"].median()
build_year_min = buildings_in_district.groupby("NAME")["BAUJAHR"].min()
build_year_max = buildings_in_district.groupby("NAME")["BAUJAHR"].max()

# Combine all statistics into a single table
build_year_stats = pd.DataFrame({
    "build_year_median": build_year_median,
    "build_year_min": build_year_min,
    "build_year_max": build_year_max
}).reset_index()

# Join the results to the districts layer on NAME
district_stats = district_stats.merge(build_year_stats, on="NAME", how="left")

_Here we group on the NAME field. In practice, group on a unique identifier instead — names repeat and get spelled differently._


Check that the new columns are in place:


In [ ]:
# Preview the first few rows
district_stats[["build_year_mean", "build_year_median", "build_year_max", "build_year_min"]].head()

Now all four statistics on separate maps, side by side.


## 3. Choropleth Maps

Let's produce four maps and compare the statistics side by side.

The usual advice for comparing maps is to put them on a **shared colour scale**, so that the same colour means the same value everywhere. We will follow that advice first, and then look hard at the result.


### 3.1. Defining the Metrics


We define the list of metrics to display. For each one, we specify:

- the field name in the data
- the map title


In [ ]:
metrics = [
    ("build_year_mean",   "Mean year of construction"),
    ("build_year_median", "Median year of construction"),
    ("build_year_min",    "Earliest year of construction"),
    ("build_year_max",    "Latest year of construction"),
]

### 3.2. Setting a Shared Colour Scale


We find the minimum and maximum values across all four metrics, so that every map is drawn on the same range.


In [ ]:
# Shared value range for all maps
vmin = district_stats[[m for m, _ in metrics]].min().min()
vmax = district_stats[[m for m, _ in metrics]].max().max()

### 3.3. Creating the Maps — Take 1

One map per metric, in a single row, on the shared colour scale.


In [ ]:
# Create a figure with 4 subplots in a single row
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)

# Set up the colour scale
norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)  # normalise values
cmap = mpl.colormaps["viridis"]  # colour palette

# Plot a map for each metric
for ax, (metric, title) in zip(axes, metrics):
    district_stats.plot(
        column=metric,      # metric to display
        ax=ax,              # target subplot
        cmap=cmap,          # colour palette
        norm=norm,          # shared value range
        linewidth=0.5,
        edgecolor="gray"
    )
    ax.set_title(title, fontsize=12)  # map title
    ax.axis("off")  # hide axes

# Add a shared colour bar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)

cbar = fig.colorbar(
    sm,
    ax=axes.tolist(),
    orientation="horizontal",
    fraction=0.05,
    pad=0.02
)
cbar.set_label("Year of construction", fontsize=12)

# Display the result
plt.show()

Look at what came out. **Three of the four maps are effectively blank** — the mean, the median and the latest year are each a single flat colour, and only the earliest year shows any structure at all.

The shared scale is the cause. `build_year_min` reaches back to 1180, so the scale has to span **831 years**, while the mean varies across the districts by only 66 — about **8 %** of the bar. Every district lands in the same slice of the colour ramp, and the differences we computed so carefully become invisible.

The advice was not wrong; it was applied too widely. A shared scale lets you compare maps **whose values share a range**. Mean and median do share one. The earliest and latest years do not — not with each other, and not with the averages.

So we draw them again, this time giving each metric its own scale.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)

for ax, (metric, title) in zip(axes, metrics):
    district_stats.plot(
        column=metric,
        ax=ax,
        cmap="viridis",
        legend=True,
        legend_kwds={"orientation": "horizontal", "shrink": 0.8},
        linewidth=0.5,
        edgecolor="gray"
    )
    ax.set_title(title, fontsize=12)
    ax.axis("off")

plt.show()

Now every map reads — but note what has been traded away. Each has its own colour bar, so the same green means a different year in each panel, and the maps can no longer be compared by colour at a glance. They have to be read one at a time, through their legends.

That is the choice, and there is no way around it: **a shared scale compares, an individual scale reveals.** Which one is right depends on whether the quantities genuinely belong on the same axis. Here, mean and median do — try them on a shared scale of their own, and the comparison works.

## Summary


In this section, we learned how to aggregate **attribute data** from a spatial dataset and analyse it by spatial unit.

We covered:

- how to use a spatial join to assign each feature to a spatial unit;
- how to group data by spatial unit and compute summary statistics;
- and that a shared colour scale is a tool with a condition attached — it compares maps whose values share a range, and flattens them when they do not.

In this example, we analysed the year of construction of buildings. The same approach applies to any numeric attribute data.
